# 1. Imports

## 1.1. Libraries

In [ ]:
import pandas as pd
import numpy as np
import requests
from sklearn.model_selection import TimeSeriesSplit
from lightgbm import LGBMRegressor
from sklearn.metrics import (mean_absolute_error, mean_squared_error)



## 1.2. Loading Data

In [323]:
df_sell_in = pd.read_excel('Business_Case_Data_Set.xlsx', sheet_name='Table 2 - Sell In')
df_external_var = pd.read_excel('Business_Case_Data_Set.xlsx', sheet_name = 'Table 1 - External Variables')
df_data_dic = pd.read_excel('Business_Case_Data_Set.xlsx', sheet_name = 'Table 0 - Data Dictionary')

# 2. Data Description

In [324]:
#Join the dfs:

df = pd.merge(df_external_var, df_sell_in, how='left', on=['Week', 'Product', 'Channel'])
df.head(5)

,Week,Product,Channel,Price_per_kg_USD,Numeric_Distribution,Weighted_Distribution,Advertising_Investment_USD,Promotion_Investment_USD,Promotion_Type,Sell_In_Tons
0,2023-07-03,Natural Juice 1L,Supermarkets,2.47,81.3,86.1,7592,1968,NaN,5.592
1,2023-07-03,Natural Juice 1L,Traditional,2.40,57.2,44.0,2962,1818,Bundle,3.278
2,2023-07-03,Natural Juice 1L,E-commerce,2.58,52.1,61.4,6245,3719,Discount,0.410
3,2023-07-03,Flavored Water 500ml,Supermarkets,1.86,68.4,74.6,7711,1795,Discount,3.531
4,2023-07-03,Flavored Water 500ml,Traditional,1.86,56.2,53.6,3409,2465,Discount,2.134


In [325]:
#Separate the traning and predict df
df_pred = df.loc[df['Sell_In_Tons'].isna()].copy()
df_train = df.loc[~df['Sell_In_Tons'].isna()].copy()

In [326]:
# Verify the sell in df
df_sell_in.head(10)

,Week,Product,Channel,Sell_In_Tons
0,2023-07-03,Natural Juice 1L,Supermarkets,5.592
1,2023-07-03,Natural Juice 1L,Traditional,3.278
2,2023-07-03,Natural Juice 1L,E-commerce,0.410
3,2023-07-03,Flavored Water 500ml,Supermarkets,3.531
4,2023-07-03,Flavored Water 500ml,Traditional,2.134
5,2023-07-03,Flavored Water 500ml,E-commerce,0.250
6,2023-07-03,Energy Drink 350ml,Supermarkets,3.350
7,2023-07-03,Energy Drink 350ml,Traditional,2.952
8,2023-07-03,Energy Drink 350ml,E-commerce,0.555
9,2023-07-03,Whole Grain Crackers 200g,Supermarkets,1.325


* Every line is the Sell In in tons, by week, per product in each Channel

In [327]:
# Verify the External Values df
df_external_var.head(5)

,Week,Product,Channel,Price_per_kg_USD,Numeric_Distribution,Weighted_Distribution,Advertising_Investment_USD,Promotion_Investment_USD,Promotion_Type
0,2023-07-03,Natural Juice 1L,Supermarkets,2.47,81.3,86.1,7592,1968,NaN
1,2023-07-03,Natural Juice 1L,Traditional,2.40,57.2,44.0,2962,1818,Bundle
2,2023-07-03,Natural Juice 1L,E-commerce,2.58,52.1,61.4,6245,3719,Discount
3,2023-07-03,Flavored Water 500ml,Supermarkets,1.86,68.4,74.6,7711,1795,Discount
4,2023-07-03,Flavored Water 500ml,Traditional,1.86,56.2,53.6,3409,2465,Discount


In [328]:
# Check the Variables definition and type:
df_data_dic

,Table,Column,Description,Data_Type
0,Table 1,Week,Week start date (Monday),Date
1,Table 1,Product,Product name,String
2,Table 1,Channel,Sales channel,String
3,Table 1,Price_per_kg_USD,Price per kilogram in USD,Float
4,Table 1,Numeric_Distribution,Numeric distribution (%),Float
5,Table 1,Weighted_Distribution,Weighted distribution (%),Float
6,Table 1,Advertising_Investment_USD,Advertising investment in USD,Float
7,Table 1,Promotion_Investment_USD,Promotion investment in USD,Float
8,Table 1,Promotion_Type,"Type of promotion (None, Discount, BOGO, Bundle)",String
9,Table 2,Week,Week start date (Monday),Date


In [329]:
print('Number of Rows of External Variables df: {}'.format(df_external_var.shape[0]))
print('Number of Cols of External Variables df: {}'.format(df_external_var.shape[1]))

Number of Rows of External Variables df: 3510
Number of Cols of External Variables df: 9


In [330]:
# Columns types information:
df_external_var.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3510 entries, 0 to 3509
Data columns (total 9 columns):
 #   Column                      Non-Null Count  Dtype  
---  ------                      --------------  -----  
 0   Week                        3510 non-null   object 
 1   Product                     3510 non-null   object 
 2   Channel                     3510 non-null   object 
 3   Price_per_kg_USD            3510 non-null   float64
 4   Numeric_Distribution        3510 non-null   float64
 5   Weighted_Distribution       3510 non-null   float64
 6   Advertising_Investment_USD  3510 non-null   int64  
 7   Promotion_Investment_USD    3510 non-null   int64  
 8   Promotion_Type              1910 non-null   object 
dtypes: float64(3), int64(2), object(4)
memory usage: 246.9+ KB


## 1.1. Sell In/Train Description

In [331]:
print('Number of Rows of Sell In df: {}'.format(df_sell_in.shape[0]))
print('Number of Cols of Sell In df: {}'.format(df_sell_in.shape[1]))

Number of Rows of Sell In df: 2340
Number of Cols of Sell In df: 4


In [332]:
print('Number of Rows of train df: {}'.format(df_train.shape[0]))
print('Number of Cols of train df: {}'.format(df_train.shape[1]))

Number of Rows of train df: 2340
Number of Cols of train df: 10


In [333]:
# Columns types information:
df_sell_in.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2340 entries, 0 to 2339
Data columns (total 4 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   Week          2340 non-null   object 
 1   Product       2340 non-null   object 
 2   Channel       2340 non-null   object 
 3   Sell_In_Tons  2340 non-null   float64
dtypes: float64(1), object(3)
memory usage: 73.3+ KB


* The date period is in the following format: YYYY-MM-DD, and represents the first day of the week (monday)

In [334]:
#Transform Week in Date Time
df_sell_in['Week'] = pd.to_datetime(df_sell_in['Week'])
df_external_var['Week'] = pd.to_datetime(df_external_var['Week'])
df_train['Week'] = pd.to_datetime(df_train['Week'])
df_pred['Week'] = pd.to_datetime(df_pred['Week'])

In [335]:
df_sell_in.head(5)

,Week,Product,Channel,Sell_In_Tons
0,2023-07-03,Natural Juice 1L,Supermarkets,5.592
1,2023-07-03,Natural Juice 1L,Traditional,3.278
2,2023-07-03,Natural Juice 1L,E-commerce,0.410
3,2023-07-03,Flavored Water 500ml,Supermarkets,3.531
4,2023-07-03,Flavored Water 500ml,Traditional,2.134


In [336]:
week_count = df_sell_in['Week'].nunique()
week_first = df_sell_in['Week'].min()
week_last = df_sell_in['Week'].max()
week_diff =  ((week_last.year - week_first.year) * 12 + (week_last.month - week_first.month))

print(f'The historic data starts in {week_first} and ends in {week_last}, with a total of {week_count} weeks and {week_diff} months.')

The historic data starts in 2023-07-03 00:00:00 and ends in 2026-06-22 00:00:00, with a total of 156 weeks and 35 months.


In [337]:
#Check NA
df_sell_in.isna().sum()

Week            0
Product         0
Channel         0
Sell_In_Tons    0
dtype: int64

In [338]:
#Check NA
df_train.isna().sum()

Week                             0
Product                          0
Channel                          0
Price_per_kg_USD                 0
Numeric_Distribution             0
Weighted_Distribution            0
Advertising_Investment_USD       0
Promotion_Investment_USD         0
Promotion_Type                1047
Sell_In_Tons                     0
dtype: int64

In [339]:
#Check NA
df_external_var.isna().sum()

Week                             0
Product                          0
Channel                          0
Price_per_kg_USD                 0
Numeric_Distribution             0
Weighted_Distribution            0
Advertising_Investment_USD       0
Promotion_Investment_USD         0
Promotion_Type                1600
dtype: int64

## 1.2. Pred df Description

In [340]:
week_count = df_external_var['Week'].nunique()
week_first = df_external_var['Week'].min()
week_last = df_external_var['Week'].max()
week_diff =  ((week_last.year - week_first.year) * 12 + (week_last.month - week_first.month))

print(f'The historic data starts in {week_first} and ends in {week_last}, with a total of {week_count} weeks and {week_diff} months.')

The historic data starts in 2023-07-03 00:00:00 and ends in 2027-12-20 00:00:00, with a total of 234 weeks and 53 months.


In [341]:
week_count = df_pred['Week'].nunique()
week_first = df_pred['Week'].min()
week_last = df_pred['Week'].max()
week_diff =  ((week_last.year - week_first.year) * 12 + (week_last.month - week_first.month))

print(f'The historic data starts in {week_first} and ends in {week_last}, with a total of {week_count} weeks and {week_diff} months.')

The historic data starts in 2026-06-29 00:00:00 and ends in 2027-12-20 00:00:00, with a total of 78 weeks and 18 months.


## 2.1. Fill Na

In [342]:
df.isna().sum()

Week                             0
Product                          0
Channel                          0
Price_per_kg_USD                 0
Numeric_Distribution             0
Weighted_Distribution            0
Advertising_Investment_USD       0
Promotion_Investment_USD         0
Promotion_Type                1600
Sell_In_Tons                  1170
dtype: int64

In [343]:
df_train['Promotion_Type'] = df_train['Promotion_Type'].fillna('unidentified')
df_pred['Promotion_Type'] = df_pred['Promotion_Type'].fillna('unidentified')

# 5. Data Transformation

In [344]:
df_train['Promotion_Type'].unique()

array(['unidentified', 'Bundle', 'Discount', 'BOGO'], dtype=object)

In [345]:
# Promotion_Type encoding
encoding = {
    'unidentified':0,
    'Discount':1,
    'BOGO':2,
    'Bundle':3
}

df_train['Promotion_Type'] = df_train['Promotion_Type'].map(encoding)

## 5.2. Deal with duplicates

In [346]:
df_train[df_train[['Week', 'Product', 'Channel']].duplicated(keep=False)]

,Week,Product,Channel,Price_per_kg_USD,Numeric_Distribution,Weighted_Distribution,Advertising_Investment_USD,Promotion_Investment_USD,Promotion_Type,Sell_In_Tons


In [347]:
df_train = df_train.drop_duplicates(subset=['Week', 'Product', 'Channel']).reset_index(drop=True)

# 4. Features Engineering

* **Lag_1w:** Sell-In volume lagged by 1 week
* **Lag_4w:**	Sell-In volume lagged by 4 weeks
* **Rolling_Mean_4w:** Rolling mean of Sell-In over last 4 weeks
* **Rolling_Std_4w:**	Rolling standard deviation of Sell-In over last 4 weeks
* **Price_Change_Pct:** Week-over-week percentage change in Price per kg
* **Promo_Flag:**	Binary flag: 1 if Promotion_Type != None, else 0
* **Month_Sin:** Sine transformation of month number (cyclical encoding)
* **Month_Cos:** Cosine transformation of month number (cyclical encoding)
* **Holiday_Flag:** Binary flag: 1 if week contains a national holiday, else 0
* **Interaction_Price_Promo:** Interaction term: Price_per_kg_USD * Promo_Flag (captures joint effect of price and promotion)


In [348]:
# Lag_1w and Lag_4w
df_train = df_train.sort_values(["Channel", "Product", "Week"]).reset_index(drop=True)

df_train["Lag_1w"] = (df_train.groupby(["Channel", "Product"])["Sell_In_Tons"].shift(1))
df_train["Lag_2w"] = (df_train.groupby(["Channel", "Product"])["Sell_In_Tons"].shift(2))
df_train["Lag_3w"] = (df_train.groupby(["Channel", "Product"])["Sell_In_Tons"].shift(3))
df_train["Lag_4w"] = (df_train.groupby(["Channel", "Product"])["Sell_In_Tons"].shift(4))


df_pred = df_pred.sort_values(["Channel", "Product", "Week"]).reset_index(drop=True)

df_pred["Lag_1w"] = (df_pred.groupby(["Channel", "Product"])["Sell_In_Tons"].shift(1))
df_pred["Lag_2w"] = (df_pred.groupby(["Channel", "Product"])["Sell_In_Tons"].shift(2))
df_pred["Lag_3w"] = (df_pred.groupby(["Channel", "Product"])["Sell_In_Tons"].shift(3))
df_pred["Lag_4w"] = (df_pred.groupby(["Channel", "Product"])["Sell_In_Tons"].shift(4))

In [349]:
# Rolling_Mean_4w

df_train['Rolling_Mean_4w'] = df_train[['Lag_1w', 'Lag_2w', 'Lag_3w', 'Lag_4w']].mean(axis=1)

In [350]:
# Rolling_Std_4w

df_train['Rolling_Std_4w'] = df_train[['Lag_1w', 'Lag_2w', 'Lag_3w', 'Lag_4w']].std(axis=1)

In [351]:
# Price_Change_Pct
df_train = df_train.sort_values(["Channel", "Product", "Week"])

df_train["Price_Change_Pct"] = (df_train.groupby(["Channel", "Product"])["Price_per_kg_USD"].pct_change())

In [352]:
# Promo_Flag
df_train['Promo_Flag'] = np.where(df_train['Promotion_Type'] != 0, 1, 0)

In [353]:
# Month_Sin and Month_Cos
df_train["Month"] = df_train["Week"].dt.month
df_train['Month_Sin'] = df_train['Month'].apply( lambda x: np.sin( x * ( 2. * np.pi/12 )) )
df_train['Month_Cos'] = df_train['Month'].apply( lambda x: np.cos( x * ( 2. * np.pi/12 )) )

In [354]:
# Holiday_Flag


In [355]:
# CREATE DF WITH HOLLIDAYS IN ECUADOR
years = range(2023, 2027)
country = "EC"

all_holidays = []

for year in years:
    url = f"https://date.nager.at/api/v3/PublicHolidays/{year}/{country}"
    response = requests.get(url)
    response.raise_for_status()

    data = response.json()  # lista de dicionários

    for item in data:
        item["year"] = year
        all_holidays.append(item)

df_holidays = pd.DataFrame(all_holidays)
df_holidays["date"] = pd.to_datetime(df_holidays["date"])

In [356]:
#CREATE DICTIONARY OF WEEK
df_calendar = pd.DataFrame({
    "Date": pd.date_range(
        start="2023-07-03",
        end="2027-12-20",
        freq="D"
    )
})

df_calendar["Week_Number"] = (
    ((df_calendar["Date"] - pd.Timestamp("2023-07-03")).dt.days // 7) + 1
)

df_calendar["Week_Start"] = (
    pd.Timestamp("2023-07-03") +
    pd.to_timedelta((df_calendar["Week_Number"] - 1) * 7, unit="D")
)

df_calendar["Day_Name"] = df_calendar["Date"].dt.day_name()

In [357]:
#Identify the week number in both dfs (holidays and train)
df_train = df_train.merge(df_calendar[['Date', 'Week_Number']], how='left', left_on='Week', right_on='Date').drop(columns='Date')
df_holidays = df_holidays.merge(df_calendar[['Date', 'Week_Number']], how='left', left_on='date', right_on='Date').drop(columns='Date')
df_holidays['Week_Number'] = df_holidays['Week_Number'].astype("Int64")
df_holidays['Holiday'] = 1


In [ ]:
df_holidays = (df_holidays.groupby("Week_Number", as_index=False)["Holiday"].max())

,Week_Number,Holiday
0,6,1
1,15,1
2,18,1
3,26,1
4,27,1
5,33,1
6,39,1
7,44,1
8,47,1
9,58,1


In [359]:
#Identify the holiday weeks in the df_train

df_train = df_train.merge(df_holidays[['Week_Number', 'Holiday']], how='left', on='Week_Number')
df_train['Holiday'] = df_train['Holiday'].fillna(0).astype("Int64")
df_train.rename(columns={"Holiday": "Holiday_Flag"}, inplace=True)

In [360]:
df_pred = df_pred.merge(df_calendar[['Date', 'Week_Number']], how='left', left_on='Week', right_on='Date').drop(columns='Date')


In [361]:
#Interaction_Price_Promo
df_train['Interaction_Price_Promo'] = df_train['Price_per_kg_USD'] * df_train['Promo_Flag']

# 5. Exploratory Data Analysis

* Quanto foi vendido por produto e channel ao longo do tempo ? (Gráfico de linhas)
* 

# 6. Product/Channel Filter

In [362]:
keys = (df_train[['Product', 'Channel']]).drop_duplicates().reset_index(drop=True)
keys

,Product,Channel
0,Cereal Bar 50g,E-commerce
1,Energy Drink 350ml,E-commerce
2,Flavored Water 500ml,E-commerce
3,Natural Juice 1L,E-commerce
4,Whole Grain Crackers 200g,E-commerce
5,Cereal Bar 50g,Supermarkets
6,Energy Drink 350ml,Supermarkets
7,Flavored Water 500ml,Supermarkets
8,Natural Juice 1L,Supermarkets
9,Whole Grain Crackers 200g,Supermarkets


In [363]:
df_train2 = df_train.copy()

In [486]:
df_train = df_train2[(df_train2['Product'] == 'Cereal Bar 50g') & (df_train2['Channel'] == 'E-commerce')]
df_train = df_train.reset_index(drop=True)

# 7. Features Selection

In [487]:
df_train.columns

Index(['Week', 'Product', 'Channel', 'Price_per_kg_USD',
       'Numeric_Distribution', 'Weighted_Distribution',
       'Advertising_Investment_USD', 'Promotion_Investment_USD',
       'Promotion_Type', 'Sell_In_Tons', 'Lag_1w', 'Lag_2w', 'Lag_3w',
       'Lag_4w', 'Rolling_Mean_4w', 'Rolling_Std_4w', 'Price_Change_Pct',
       'Promo_Flag', 'Month', 'Month_Sin', 'Month_Cos', 'Week_Number',
       'Holiday_Flag', 'Interaction_Price_Promo'],
      dtype='object')

In [488]:
df_train = df_train.drop(columns=['Lag_2w', 'Lag_3w', 'Month', 'Week_Number', 'Product', 'Channel', 'Week'])

# 8. Machine Learnin Modeling 

* I decided using the method of the rolling backtest, using 52 weeks (1 year) as window, i.e. a whole year.

## 8.1. Prepare target feature

In [489]:
target = 'Sell_In_Tons'

features = [c for c in df_train.columns if c != target]

X = df_train[features]
y = df_train[target]

## 8.2. Rolling Backtest

In [490]:
tscv = TimeSeriesSplit(
    n_splits=2,
    test_size=52
)

In [491]:
for train_idx, val_idx in tscv.split(X):

    X_train = X.iloc[train_idx]
    y_train = y.iloc[train_idx]

    X_val = X.iloc[val_idx]
    y_val = y.iloc[val_idx]

    print(len(X_train), len(X_val))

52 52
104 52


## 8.3. LGBM training

In [492]:

model = LGBMRegressor(
    objective="regression",
    learning_rate=0.1,
    n_estimators=500,
    max_depth=12,
    num_leaves=256,
    min_child_samples=5,
    subsample=0.7,
    colsample_bytree=0.7,
    reg_alpha=0.0,
    reg_lambda=0.0,
    min_split_gain=0.0,
    boosting_type='gbdt',
    metric='mape',
    random_state=42,
    n_jobs=-1,
    verbose=-1
)

model.fit(X_train, y_train)

LGBMRegressor(colsample_bytree=0.7, max_depth=12, metric='mape',
              min_child_samples=5, n_estimators=500, n_jobs=-1, num_leaves=256,
              objective='regression', random_state=42, subsample=0.7,
              verbose=-1)

## 8.4. Predict

In [493]:
pred = model.predict(X_val)

## 8.5. Metrics Calculation

In [ ]:

mae = mean_absolute_error(y_val, pred)

rmse = np.sqrt(mean_squared_error(y_val, pred))

mape = np.mean(np.abs((y_val - pred) / y_val)) * 100

In [495]:
results = []


In [496]:
results.append({
    "MAE": mae,
    "RMSE": rmse,
    "MAPE": mape
})

In [497]:
pd.DataFrame(results)

,MAE,RMSE,MAPE
0,0.141084,0.175169,14.980831


## 8.6. 

# 9. Hyperparameter Fine Tuning

# 10. Translation and Interpretation of Performance